# Filter sampling rate: the minimum rate and how it is set

Works out the minimum rate at which the SoC filter must run, by three criteria, and builds the figures and tables of `final_report/sampling_rate_section/sampling_rate.md`. Run all cells top to bottom; the last cell copies the figures into `sampling_rate_section/figures/`.

- **Criterion 1, model time constant.** Δt ≤ τ₁/10, with τ₁ = R₁C₁ from the identified 2RC tables, at its smallest over SoC.
- **Criterion 2, current bandwidth.** f_s ≥ f_I,max / 0.075: the frequencies of interest stay at or below 15 % of the Nyquist frequency (Smyth & Wu). f_I,max comes from the PSD of the logged battery current on the operational runs.
- **Criterion 3, rate sweep.** The tuned EKF and coulomb counting on each recording, reduced to 20 … 0.02 Hz by block mean, block median and point sampling, compared with the same filter on the block mean of 10 logged samples (the server's window, about 115 Hz on the ADS8688).
- **Code.** `soc_eval.py`, the "sampling rate" block at the end of the file. The sweep uses the frozen tuned settings (`settings(battery, model)`), with the battery current as logged, so it isolates the sampling.
- **Outputs.** Figures go to `final_report/figures/rate_*.{png,pdf}`. Tables go to `final_report/tables/res_rate_*.{csv,md}`; the full sweep is `tables/rate_sweep.csv`.
- **Run time.** About 20–30 minutes, most of it the reference EKF at about 115 Hz. With `RERUN = False`, section 2 reloads the saved sweep and spectra instead.

In [ ]:
import gc
import sys
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'final_report' / 'soc_eval.py').exists()) / 'final_report'))
import soc_eval as se

se.apply_style()
pd.set_option('display.precision', 4)
pd.set_option('display.width', 200)
models = {b: se.load_model(c['model']) for b, c in se.BATTERIES.items()}
NUM = {}
RERUN = True        # False: reuse tables/rate_sweep.csv and tables/rate_psd.pkl from an earlier run

## 1. Criterion 1: the fast RC time constant (Tables S1 and S2)

The EKF propagates each RC voltage with the exact decay factor a = exp(−Δt/τ) and input gain R(1 − a), under a zero-order hold on the current over Δt (`ekf_batch`, `filter.js`). The propagation therefore has no discretisation error of its own. The τ₁/10 rule is kept for two reasons: it bounds the zero-order-hold error, and it gives the filter about ten samples per fast relaxation. Table S2 shows the error a first-order factor 1 − Δt/τ would have had.

In [ ]:
TAU = se.time_constant_table(models)
display(Markdown(se.md_table(TAU, 'rate_time_constants', floatfmt='.2f')))
TAU_RATE = TAU['f_s min (Hz)'].max()                 # the battery with the smaller tau1 binds
TAU_BATTERY = TAU['f_s min (Hz)'].idxmax()

server_dt = {'LiFePO4': 10 * 0.866e-3, 'LiNMC': 10 * 32e-3}     # server window: 10 ADS8688 samples / 10 ADS1115 samples
ratios = {'Δt/τ₁ = 0.1': 0.1, 'Δt/τ₁ = 0.05': 0.05, 'Δt/τ₁ = 0.01': 0.01}
ratios |= {f'server window, {b} (Δt = {1e3 * d:.1f} ms)': d / TAU.loc[b, 'τ1 min (s)'] for b, d in server_dt.items()}
EULER = pd.DataFrame({'Δt/τ₁': ratios.values(), 'first-order factor error (%)': 100 * se.euler_error(list(ratios.values()))},
                     index=list(ratios))
display(Markdown(se.md_table(EULER, 'rate_euler', floatfmt='.3g')))
NUM['criterion 1'] = f"{TAU_BATTERY}: τ1 min {TAU.loc[TAU_BATTERY, 'τ1 min (s)']:.2f} s -> Δt <= {TAU.loc[TAU_BATTERY, 'Δt max (s)']:.2f} s, f_s >= {TAU_RATE:.3f} Hz"
print(NUM['criterion 1'])

## 2. Load every run, spectra and sweep (the slow part)

Each run is loaded at its logged rate, used, and released before the next, to keep memory use to one run at a time.
- The PSD (criterion 2) is computed on the operational runs only. The controlled runs are 10-minute constant-current steps, not an operating load.
- The sweep (criterion 3) covers all six runs. On the LiNMC controlled run (ADS1115, 32 ms) the reference is the logged rate itself, and only rates up to half of it are swept.

In [ ]:
SWEEP_FILE, PSD_FILE = se.TABLE_DIR / 'rate_sweep.csv', se.TABLE_DIR / 'rate_psd.pkl'
if RERUN or not SWEEP_FILE.exists():
    PSD, sweeps = {}, []
    for run in se.SWEEP_RUNS:
        r = se.RUNS[run]
        t0 = time.time()
        raw = se.load_sweep_run(run)
        if r['kind'] == 'operational':
            PSD[run] = se.current_psd(raw)
        sweeps.append(se.run_sweep(run, models[r['battery']], raw))
        print(f"{run}: {len(raw):,} logged samples, {raw.t.iloc[-1] / 3600:.2f} h, {time.time() - t0:.0f} s")
        del raw
        gc.collect()
    SW = pd.concat(sweeps, ignore_index=True)
    se.save_table(SW, 'rate_sweep', index=False)
    pd.to_pickle(PSD, PSD_FILE)
else:
    SW, PSD = pd.read_csv(SWEEP_FILE), pd.read_pickle(PSD_FILE)

## 3. Criterion 2: the bandwidth of the battery current (Table S3, Figure S1)

- **Load cycle.** Welch PSD of the battery current over every connected sample, switching included, with 2¹⁷-sample segments (about 114 s, a 0.009 Hz resolution).
- **Noise floor.** The same PSD over rests where every current stays below 0.5 A for at least 10 s (2¹⁴-sample segments).
- **f_I,max.** The frequency below which 99 % of the load-driven power (load cycle minus floor) lies.

The rule f_s ≥ f_I,max / 0.075 is a point-sampling rule: it bounds the integration error when one sample stands for the whole interval.

In [ ]:
BW = se.bandwidth_table(PSD)
display(Markdown(se.md_table(BW, 'rate_bandwidth', floatfmt='.3g')))
fig = se.plot_current_psd(PSD, 'Battery current PSD on the operational runs, against its rest noise floor')
se.save(fig, 'rate_psd')
plt.show()

# The two discharge runs switch the load; the 12 Jun runs are charging near full with almost no load switching, so
# their 99 % bandwidth is set by low-level ripple (see the rms columns).
DISCHARGE_RUNS = ['LiNMC Battery Discharge Test', 'LiFePO4 Battery Discharge Test']
F_I_MAX = max(PSD[r]['f_bw'] for r in DISCHARGE_RUNS)
NYQ_RATE = F_I_MAX / (se.NYQUIST_SHARE / 2)
NUM['criterion 2'] = (f'f_I,max {F_I_MAX:.3g} Hz (largest f99 of the discharge runs) -> f_s >= {NYQ_RATE:.3g} Hz '
                      'when point sampled')
NUM['criterion 2, charging runs'] = '; '.join(f"{se.RUNS[r]['short']}: f99 {PSD[r]['f_bw']:.0f} Hz, ripple above 10 Hz "
                                              f"{PSD[r]['rms_above_10Hz']:.3f} A rms" for r in PSD if r not in DISCHARGE_RUNS)
print(NUM['criterion 2']); print(NUM['criterion 2, charging runs'])

## 4. Criterion 3: the rate sweep (Tables S4 and S5, Figure S2)

For each run, rate and decimation, the sweep reports the largest and RMS difference of the EKF SoC from the reference, and the largest difference of coulomb counting from the integral of every logged sample. All are taken over connected samples. The tolerance is `RATE_TOL_PP` = 0.05 pp. The EKF's own RMS error against the characterization SoC is 0.13 pp (LiFePO4) and 0.25 pp (LiNMC), so the tolerance keeps the sampling error to well under half of that.

Three rules keep the sweep measuring the rate and nothing else:
- **Fitted range on controlled runs.** Controlled runs are scored over the same characterization-SoC range and sessions as the rest of the evaluation (LiFePO4 11–84 %, LiNMC 3.4–92.2 % in the first three sessions). Outside that range, two sweep values come from parts of the run outside the fitted model:
  - LiFePO4: the BMS cut-off at 2.61 h, where the filter clips at 0 % by an amount that depends on the step (0.43 pp at 0.5 Hz);
  - LiNMC: the first update at 99.9 % (0.054 pp at 2 Hz).
  Operational runs are scored over every connected sample.
- **Connected samples only.** Block means and medians use only samples above the 30 V cutoff, and the block current is the block's connected charge divided by its length. Without this rule, a 2.6 s logger dropout (0 V) on the LiNMC controlled run averaged into a 20 s block as 38.4 V. That is above the cutoff, so it reached the filter as a false voltage and moved the SoC by 1.07 pp.
- **Discrete events.** The max |ΔSoC| skips records within `EVENT_WINDOW_S` = 10 s (or two steps, if longer) of a reference step larger than `EVENT_PP` = 0.05 pp. At such a step the voltage update switches on at a steep part of the curve, for example −1.34 pp at 17.8 % on the LiFePO4 controlled run.
  - Another input makes the same jump a little earlier or later. At the filter rate, the shift is up to one step. With the median, the shift is a few seconds: its 0.06 pp charge difference moves the threshold crossing by about 3 s at 26 A.
  - Such a shift is a timing difference, not a SoC difference.
  - The RMS and the end value keep every record, and the `near_jump` column counts the skipped records.

- **Table S4.** The worst run at each rate.
- **Table S5.** The lowest rate that keeps every run within the tolerance, there and at every higher rate.
- **Reference rows.** These show what the server's own median window costs against a mean at the same rate.

In [ ]:
s = SW[~SW.reference]
WORST = (s.groupby(['rate_hz', 'decimation'])[['ekf_max_pp', 'ekf_rms_pp', 'cc_max_pp']].max()
         .unstack('decimation').sort_index(ascending=False))
WORST.columns = [f'{c} ({se.DECIMATION[d]})' for c, d in WORST.columns]
display(Markdown(se.md_table(WORST, 'rate_sweep_worst', floatfmt='.4f')))

REF = SW[SW.reference & (SW.decimation != 'mean')].assign(run=lambda d: d.run.map(lambda r: se.RUNS[r]['short']))
display(Markdown(se.md_table(REF[['run', 'rate_hz', 'decimation', 'ekf_max_pp', 'cc_max_pp']], 'rate_reference_rows',
                             floatfmt='.4f', index=False)))
JUMPS = (SW[SW.decimation == 'mean'].pivot_table(index='rate_hz', columns='run', values='near_jump', aggfunc='first')
         .sort_index(ascending=False).fillna(0).astype(int))
JUMPS.columns = [se.RUNS[c]['short'] for c in JUMPS.columns]
display(Markdown(se.md_table(JUMPS, 'rate_near_jump', floatfmt='.0f')))

ABS = (SW[SW.abs_rms_pp.notna() & (SW.decimation == 'mean')]
       .pivot_table(index='rate_hz', columns='run', values='abs_rms_pp').sort_index(ascending=False))
ABS.columns = [se.RUNS[c]['short'] for c in ABS.columns]
display(Markdown(se.md_table(ABS, 'rate_controlled_abs', floatfmt='.4f')))

MIN = se.min_rates(SW)
display(Markdown(se.md_table(MIN, 'rate_min', floatfmt='.4g')))
fig = se.plot_rate_sweep(SW, TAU_RATE, NYQ_RATE, 'SoC change from the filter rate alone, against the 10-sample block-mean reference')
se.save(fig, 'rate_sweep')
plt.show()

## 5. The minimum rate (Table S6)

The minimum rate is the stricter of criterion 1 and the sweep for the decimation the filter actually uses. Criterion 2 applies only if the current is point sampled.

In [ ]:
sweep_mean = MIN.loc['block mean', 'minimum rate (Hz)']
sweep_point = MIN.loc['point sample', 'minimum rate (Hz)']
F_MIN = max(TAU_RATE, sweep_mean)
SUMMARY = pd.DataFrame({
    'minimum f_s (Hz)': [TAU_RATE, NYQ_RATE, sweep_mean, MIN.loc['block median', 'minimum rate (Hz)'], sweep_point, F_MIN],
    'basis': [f"Δt ≤ τ₁/10, {TAU_BATTERY} τ₁ = {TAU.loc[TAU_BATTERY, 'τ1 min (s)']:.1f} s",
              f'f_s ≥ f_I,max / 0.075, f_I,max = {F_I_MAX:.2g} Hz (point sampling only)',
              f'sweep, block mean, every run within {se.RATE_TOL_PP} pp',
              f'sweep, block median, every run within {se.RATE_TOL_PP} pp',
              f'sweep, point sample, every run within {se.RATE_TOL_PP} pp',
              'minimum: stricter of criterion 1 and the block-mean sweep']},
    index=['1. time constant', '2. current bandwidth', '3. sweep (mean)', '3. sweep (median)', '3. sweep (point)', 'minimum'])
display(Markdown(se.md_table(SUMMARY, 'rate_summary', floatfmt='.3g')))
NUM['minimum'] = f'{F_MIN:.3g} Hz (Δt <= {1 / F_MIN:.2f} s)'
NUM['server rate'] = f"about {1 / (10 * 0.866e-3):.0f} Hz on the ADS8688, {1 / (10 * 32e-3):.2f} Hz on the ADS1115 run"

## Numbers quoted in the text, and copy to `sampling_rate_section/`

In [ ]:
for k, v in NUM.items():
    print(f'{k}: {v}')
out = se.SAMPLING_DIR / 'figures'
out.mkdir(parents=True, exist_ok=True)
for name in ('rate_psd', 'rate_sweep'):
    shutil.copy2(se.FIG_DIR / f'{name}.png', out / f'{name}.png')
print(f'Copied 2 figures to {out}')